In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv("data/raw/US_Accidents_March23_sampled_500k.csv")
df.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-2047758,Source2,2,2019-06-12 10:10:56,2019-06-12 10:55:58,30.641211,-91.153481,NaN,NaN,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day
1,A-4694324,Source1,2,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,38.990562,-77.399070,38.990037,-77.398282,0.056,...,False,False,False,False,False,False,Night,Night,Night,Night
2,A-5006183,Source1,2,2022-08-20 13:13:00.000000000,2022-08-20 15:22:45.000000000,34.661189,-120.492822,34.661189,-120.492442,0.022,...,False,False,False,False,True,False,Day,Day,Day,Day
3,A-4237356,Source1,2,2022-02-21 17:43:04,2022-02-21 19:43:23,43.680592,-92.993317,43.680574,-92.972223,1.054,...,False,False,False,False,False,False,Day,Day,Day,Day
4,A-6690583,Source1,2,2020-12-04 01:46:00,2020-12-04 04:13:09,35.395484,-118.985176,35.395476,-118.985995,0.046,...,False,False,False,False,False,False,Night,Night,Night,Night


In [3]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 500,000
Columns: 46


In [4]:
# Missing data analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values,
    'Data Type': missing.index.map(df.dtypes),
}).sort_values('Missing_Percentage', ascending=False)

# Show only columns with missing data
print("MISSING DATA SUMMARY")
print(missing_df[missing_df['Missing_Count'] > 0])

MISSING DATA SUMMARY
                   Column  Missing_Count  Missing_Percentage Data Type
8                 End_Lng         220377             44.0754   float64
7                 End_Lat         220377             44.0754   float64
27      Precipitation(in)         142616             28.5232   float64
21          Wind_Chill(F)         129017             25.8034   float64
26        Wind_Speed(mph)          36987              7.3974   float64
24         Visibility(mi)          11291              2.2582   float64
25         Wind_Direction          11197              2.2394    object
22            Humidity(%)          11130              2.2260   float64
28      Weather_Condition          11101              2.2202    object
20         Temperature(F)          10466              2.0932   float64
23           Pressure(in)           8928              1.7856   float64
19      Weather_Timestamp           7674              1.5348    object
42         Sunrise_Sunset           1483              0.

In [5]:
# Check the relationship between Temperature and Wind_Chill
df_both = df[df['Wind_Chill(F)'].notna() & df['Temperature(F)'].notna()]

# Calculate the difference
df_both['Temp_WC_Diff'] = df_both['Temperature(F)'] - df_both['Wind_Chill(F)']

print("Difference between Temperature and Wind_Chill:")
print(df_both['Temp_WC_Diff'].describe())
print()

# How many are exactly the same?
same_count = (df_both['Temp_WC_Diff'] == 0).sum()
print(f"Exactly the same: {same_count:,} ({same_count/len(df_both)*100:.1f}%)")
print()

# How many differ when they are not the same?
small_diff = (abs(df_both['Temp_WC_Diff']) != 0).sum()
print(f"Differ: {small_diff:,} ({small_diff/len(df_both)*100:.1f}%)")

Difference between Temperature and Wind_Chill:
count    370983.000000
mean          1.611013
std           3.460652
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          34.000000
Name: Temp_WC_Diff, dtype: float64

Exactly the same: 286,754 (77.3%)

Differ: 84,229 (22.7%)


In [6]:
print("Full duplicates:", df.duplicated().sum())

Full duplicates: 0


In [7]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['End_Lat', 'End_Lng', 'Wind_Chill(F)'])

# datetime conversion
time_cols = ['Start_Time', 'End_Time', 'Weather_Timestamp']
for col in time_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# 1. Clean and standardize case
df_clean['Wind_Direction'] = df_clean['Wind_Direction'].astype('string').str.strip().str.upper()

# 2. Define the map in UPPERCASE to match the data
wind_dir_map = {
    'NORTH': 'N',
    'SOUTH': 'S',
    'EAST': 'E',
    'WEST': 'W',
    'VARIABLE': 'VAR',
    'CALM': 'CALM'
}

# 3. Apply the replacement
df_clean['Wind_Direction'] = df_clean['Wind_Direction'].replace(wind_dir_map)

#standardize zip codes
df_clean['Zipcode'] = df_clean['Zipcode'].astype('string').str.strip()
df_clean['Zipcode'] = df_clean['Zipcode'].str[:5]
df_clean['Zipcode'] = df_clean['Zipcode'].str.zfill(5)
df_clean['Zipcode'] = df_clean['Zipcode'].replace(['00nan', 'nan', 'None', '00000'], pd.NA)

In [8]:
# Are weather features missing when Weather_Timestamp is missing?
weather_features = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 
                   'Visibility(mi)', 'Wind_Speed(mph)', 'Wind_Direction',
                   'Weather_Condition', 'Precipitation(in)']

# Rows where Weather_Timestamp is missing
wt_missing = df_clean[df_clean['Weather_Timestamp'].isnull()]

print(f"Rows with missing Weather_Timestamp: {len(wt_missing)}")
print("\nWeather features in those rows:")
for feature in weather_features:
    missing_count = wt_missing[feature].isnull().sum()
    missing_pct = (missing_count / len(wt_missing)) * 100
    print(f"  {feature:25s}: {missing_pct:5.1f}% missing")

Rows with missing Weather_Timestamp: 7674

Weather features in those rows:
  Temperature(F)           : 100.0% missing
  Humidity(%)              : 100.0% missing
  Pressure(in)             : 100.0% missing
  Visibility(mi)           : 100.0% missing
  Wind_Speed(mph)          : 100.0% missing
  Wind_Direction           : 100.0% missing
  Weather_Condition        : 100.0% missing
  Precipitation(in)        : 100.0% missing


In [9]:


# Drop rows where Start_Time or Weather_Timestamp is missing (since weather data will be missing too)
df_clean = df_clean.dropna(subset=['Start_Time','Weather_Timestamp'])



In [10]:
df_clean['Precipitation(in)'] = df_clean['Precipitation(in)'].fillna(0)

# Fill missing values: mode for categorical, median for numerical
cat_cols = df_clean.select_dtypes(include=['object', 'string']).columns
num_cols = df_clean.select_dtypes(include=['float', 'int']).columns
boolean_columns = df_clean.select_dtypes(include=['bool']).columns.tolist()


for col in boolean_columns:
    df_clean[col] = df_clean[col].astype(int)

# Fill categorical columns with mode
for col in cat_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_value = df_clean[col].mode()
        if len(mode_value) > 0:
            df_clean[col].fillna(mode_value[0], inplace=True)

# Fill numerical columns with median
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

In [11]:
# Full missing data summary after all cleaning
missing = df_clean.isnull().sum()
missing


ID                       0
Source                   0
Severity                 0
Start_Time               0
End_Time                 0
Start_Lat                0
Start_Lng                0
Distance(mi)             0
Description              0
Street                   0
City                     0
County                   0
State                    0
Zipcode                  0
Country                  0
Timezone                 0
Airport_Code             0
Weather_Timestamp        0
Temperature(F)           0
Humidity(%)              0
Pressure(in)             0
Visibility(mi)           0
Wind_Direction           0
Wind_Speed(mph)          0
Precipitation(in)        0
Weather_Condition        0
Amenity                  0
Bump                     0
Crossing                 0
Give_Way                 0
Junction                 0
No_Exit                  0
Railway                  0
Roundabout               0
Station                  0
Stop                     0
Traffic_Calming          0
T

In [12]:
print(df_clean['Severity'].value_counts().sort_index())
print("\nPercentages:")
print(df_clean['Severity'].value_counts(normalize=True).sort_index() * 100)

Severity
1      4226
2    345878
3     83412
4     11557
Name: count, dtype: int64

Percentages:
Severity
1     0.949507
2    77.712645
3    18.741195
4     2.596653
Name: proportion, dtype: float64
